In [2]:
import re
from urllib.parse import urlparse
import tldextract
import pandas as pd

In [3]:
def is_valid_domain_or_url(value):
    try:
        parsed = urlparse(value if '://' in value else 'http://' + value)
        ext = tldextract.extract(parsed.netloc or parsed.path)
        return bool(ext.domain and ext.suffix)
    except:
        return False

def classify_store(value):
    if not isinstance(value, str):
        return "Unknown"
    
    value = value.strip()

    # ✅ Vizio
    if value.lower().startswith("vizio"):
        return "Vizio store"
    
    # ✅ Samsung TV Store: starts with G followed by digits
    if re.fullmatch(r'[Gg]\d+', value):
        return "Samsung TV store"
    
    # ✅ Amazon Store: starts with B followed by alphanumerics
    if re.fullmatch(r'[Bb][A-Za-z0-9]+', value):
        return "Amazon store"
    
    # ✅ Apple / Roku / LG Store: numeric only
    if value.isdigit():
        return "Apple, Roku, LG, Philips"
    
    # ✅ Playstore: has dots but not a valid domain
    if '.' in value:
        if is_valid_domain_or_url(value):
            return "Website"
        else:
            return "Playstore"
    
    return "Unknown"


In [6]:
df = pd.read_csv(r"C:\Users\zaid\Downloads\check_bundle_id_domain.csv")
df

,domain_bundle
0,apps.infinite.dicerollsound
1,randstad-india.pissedconsumer.com
2,minnetonkaorchards.com
3,preprod.dailybulletin.com
4,leekduck.com
...,...
170869,shebudgets.com
170870,browneyedbaker.com
170871,freebiefindingmom.com
170872,footballinsider247.com


In [7]:
df["store"] = df['domain_bundle'].apply(classify_store)
df

,domain_bundle,store
0,apps.infinite.dicerollsound,Playstore
1,randstad-india.pissedconsumer.com,Website
2,minnetonkaorchards.com,Website
3,preprod.dailybulletin.com,Website
4,leekduck.com,Website
...,...,...
170869,shebudgets.com,Website
170870,browneyedbaker.com,Website
170871,freebiefindingmom.com,Website
170872,footballinsider247.com,Website


In [8]:
df.to_csv(r"C:\Users\zaid\Downloads\check_bundle_id_domain_out.csv")

In [9]:
tldextract.extract("_._tcp.9188f7b0-5c19-46a8-8697-aec18b495f03.domains._msdcs.post-gazette.com")

ExtractResult(subdomain='_._tcp.9188f7b0-5c19-46a8-8697-aec18b495f03.domains._msdcs', domain='post-gazette', suffix='com', is_private=False)

In [4]:
import pandas as pd

df = pd.read_csv(r"beeswax.csv")

df_domain = pd.DataFrame({"domain":df['domain'].unique()})
df_domain['http_client'] = "curl_cffi"
df_domain['ads_page_url'] = ""
df_domain['inventory_type'] = "ads.txt"
df_domain

,domain,http_client,ads_page_url,inventory_type
0,http://www.14news.com/story/13451864/weather-a...,curl_cffi,,ads.txt
1,http://www.abcactionnews.com,curl_cffi,,ads.txt
2,http://www.hawaiinewsnow.com,curl_cffi,,ads.txt
3,http://www.kait8.com,curl_cffi,,ads.txt
4,http://www.live5news.com/category/130322/conta...,curl_cffi,,ads.txt
...,...,...,...,...
259,https://www.bet.com,curl_cffi,,ads.txt
260,https://www.cbsnews.com,curl_cffi,,ads.txt
261,https://morganmurphymedia.com/,curl_cffi,,ads.txt
262,https://support.cwtv.com/hc/en-us/,curl_cffi,,ads.txt


In [ ]:
import tempfile
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from itertools import islice
from collections import defaultdict
import logging

logger = logging.getLogger(__name__)
BATCH_SIZE = 100


def generate_start_urls(df_chunk):
    for _, row in df_chunk.iterrows():
        yield (
            row["domain"],
            defaultdict(
                None,
                {
                    "http_client": row["http_client"],
                    "inventory_type": row["inventory_type"],
                    "ads_txt_page_url": row["ads_page_url"],
                },
            ),
        )


def read_parquet_in_batches(parquet_path, batch_size):
    parquet_file = pq.ParquetFile(parquet_path)
    print(parquet_file.num_row_groups)
    for rg in range(parquet_file.num_row_groups):
        table = parquet_file.read_row_group(rg)
        df_chunk = table.to_pandas()

        it = iter(df_chunk.iterrows())
        while True:
            batch = list(islice(it, batch_size))
            if not batch:
                break
            yield pd.DataFrame([row for _, row in batch])


def run_batched_spider(df, settings=None):
    # Create a temporary Parquet file
    with tempfile.TemporaryDirectory() as temp_dir:

        tmp_file_name = Path(temp_dir) / "domains_meta_data.parquet"

        logger.info(f"Saving Parquet temp file at {tmp_file_name}")

        # Fill and save DataFrame
        df.fillna({
            "http_client": "curl_cffi",
            "ads_page_url": "",
            "domain": "NA",
            "inventory_type": "ads.txt",
        }, inplace=True)

        table = pa.Table.from_pandas(df)
        pq.write_table(table, tmp_file_name, row_group_size=BATCH_SIZE)

        del df  # free memory early

        # Read and crawl batches
        for batch_num, df_batch in enumerate(read_parquet_in_batches(tmp_file_name, BATCH_SIZE), start=1):
            logger.info(f"Running batch {batch_num}, size: {len(df_batch)}")
            start_urls_with_meta = generate_start_urls(df_batch)
            start_urls_with_meta = list(start_urls_with_meta)
            # run_spider(start_urls_with_meta, project_setting=settings, is_test=False)
            print(f"-----------------------------------------batch running ------ {batch_num}")
            logger.info(f"Batch {batch_num} finished.")

    logger.info("All batches completed.")

run_batched_spider(df_domain)

3
100
<class 'list'>
[('http://www.14news.com/story/13451864/weather-app-information-page', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.abcactionnews.com', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.hawaiinewsnow.com', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.kait8.com', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.live5news.com/category/130322/contact-live-5', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.kctv5.com', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_url': ''})), ('http://www.walb.com', defaultdict(None, {'http_client': 'curl_cffi', 'inventory_type': 'ads.txt', 'ads_txt_page_

In [23]:
bool(int(False))
int(True)

1

In [1]:
import re
from urllib.parse import urlparse
import tldextract

# def tldextract_valid(input_str):

#     parsed = tldextract.extract(input_str)
#     main_domain = f"{parsed.domain}.{parsed.suffix}"
#     root_domain = f"http://{parsed.subdomain+'.' if parsed.subdomain else ''}{main_domain}/"
#     print(f"{parsed=}")

#     return root_domain


# def is_valid_url(url):
#     try:
#         parsed = urlparse(url)
#         return parsed.scheme in ("http", "https") and bool(parsed.netloc)
#     except Exception:
#         return False

# def is_valid_domain(domain):
#     domain_regex = re.compile(r"^(?!-)(?:[a-zA-Z0-9-]{1,63}\.)+[a-zA-Z]{2,}$")
#     return bool(domain_regex.fullmatch(domain))

# def is_valid_domain_url(input_str):
#     if input_str is None:
#         return False
#     # input_str = tldextract_valid(input_str)
#     return is_valid_url(input_str) or is_valid_domain(input_str)
#     # return tldextract_valid(input_str)


urls = ['citationmachine.com', 'citationmachine1.com/', 'www.citationmachine2.com', "www.citationmachine3.com/exit",'www.citationmachine4.com/exit/', 'http://www.citationmachine5.com/exit', 'http://www.citationmachine6.com', 'https://citationmachine7.com', 'citationmachine8.com      '," ","","news.citationmachine7.com","Missing","amhatre@airfind.com"]

# for url in urls:
#     print(url)
#     new_url = is_valid_domain_url(url)
#     print(f"{new_url=}")
#     print("---------------------")


# def is_valid_url(s):
#     # Regex pattern to match common URLs with or without schemes
#     url_pattern = re.compile(
#         r'^(https?://)?'                # optional scheme
#         r'(www\.)?'                     # optional www
#         r'([a-zA-Z0-9-]+\.)+'           # subdomains/domains
#         r'[a-zA-Z]{2,}'                 # top-level domain (e.g. .com, .in)
#         r'(/[\w\-./]*)*$',              # optional path
#         re.IGNORECASE
#     )
#     return bool(url_pattern.match(s))

# def clean_and_check_urls(url_list):
#     results = []
#     for item in url_list:
#         # Remove leading/trailing whitespace and non-printable characters
#         cleaned = ''.join(ch for ch in item.strip() if ch.isprintable())
#         if not cleaned:
#             results.append((item, "Invalid (Empty or Non-printing)"))
#         elif is_valid_url(cleaned):
#             results.append((item, "Valid URL"))
#         else:
#             results.append((item, "Not a URL"))
#     return results

def check_if_valid_url(input_string):
    # Clean the string: strip spaces and remove non-printing characters
    cleaned = ''.join(ch for ch in input_string.strip() if ch.isprintable())

    # If the cleaned string is empty
    if not cleaned:
        return (False, "Invalid (Empty or Non-printing)")

    # Regex to match URLs
    url_pattern = re.compile(
        r'^(https?://)?'                # optional scheme
        r'(www\.)?'                     # optional www
        r'([a-zA-Z0-9-]+\.)+'           # domain/subdomain
        r'[a-zA-Z]{2,}'                 # TLD like .com, .in, etc.
        r'(/[\w\-./]*)*$',              # optional path
        re.IGNORECASE
    )

    if url_pattern.match(cleaned):
        return (True, "Valid URL")
    else:
        return (False, "Not a URL")
    
for example in urls:
    is_valid, message = check_if_valid_url(example)
    print(f"'{example}': {message} -> {is_valid}")

'citationmachine.com': Valid URL -> True
'citationmachine1.com/': Valid URL -> True
'www.citationmachine2.com': Valid URL -> True
'www.citationmachine3.com/exit': Valid URL -> True
'www.citationmachine4.com/exit/': Valid URL -> True
'http://www.citationmachine5.com/exit': Valid URL -> True
'http://www.citationmachine6.com': Valid URL -> True
'https://citationmachine7.com': Valid URL -> True
'citationmachine8.com      ': Valid URL -> True
' ': Invalid (Empty or Non-printing) -> False
'': Invalid (Empty or Non-printing) -> False
'news.citationmachine7.com': Valid URL -> True
'Missing': Not a URL -> False
'amhatre@airfind.com': Not a URL -> False


In [ ]:
def is_original_or_new_domain_same(original_domain, new_domain):
    # Clean the string: strip spaces and remove non-printing characters
    original_domain = "".join(
        ch for ch in original_domain.strip() if ch.isprintable()
    )
    # Extract the new root domain from the response URL
    parsed = tldextract.extract(original_domain)
    old_root_domain = f"{parsed.domain}.{parsed.suffix}"
    # Extract the new root domain from the response URL
    parsed = tldextract.extract(new_domain)
    new_root_domain = f"{parsed.domain}.{parsed.suffix}"

    return old_root_domain == new_root_domain